In [1]:
!rm -rf diffs/*

In [29]:
import os
import difflib
from pathlib import Path
from tqdm import tqdm

ground_truth_dir = "/home/t-swsingh/proof-model/ours/verified-code-gen/benchmarks/verus-proof-synthesis/benchmarks/VeruSAGE-Bench-Spec-100/good-files/ground-truth"
tasks_dir = "/home/t-swsingh/proof-model/ours/verified-code-gen/benchmarks/verus-proof-synthesis/benchmarks/VeruSAGE-Bench-Spec-100/good-files/tasks"

for filename in tqdm(os.listdir(ground_truth_dir), total=len(os.listdir(ground_truth_dir)), desc="Processing files"):
  gt_file = os.path.join(ground_truth_dir, filename)
  task_file = os.path.join(tasks_dir, filename)
  
  with open(gt_file, 'r') as f:
    gt_lines = f.read().splitlines(keepends=True)
  with open(task_file, 'r') as f:
    task_lines = f.read().splitlines(keepends=True)
  
  diff = difflib.unified_diff(
    gt_lines, 
    task_lines, 
    fromfile=f"ground-truth/{filename}",
    tofile=f"tasks/{filename}",
    n=5,
  )
  
  diff_filename = filename.replace('.rs', '.diff')
  diff_path = os.path.join("/home/t-swsingh/proof-model/ours/verified-code-gen/benchmarks/verus-proof-synthesis/benchmarks/VeruSAGE-Bench-Spec-100/good-files/diffs", diff_filename)
  with open(diff_path, 'w') as f:
    f.write(''.join(diff))
  
  print(f"Saved diff for {diff_filename}")

Processing files: 100%|██████████| 52/52 [00:00<00:00, 521.99it/s]

Saved diff for AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__handle_external_request.diff
Saved diff for AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__handle_get_request_msg.diff
Saved diff for NO_spec__spec__utils__map_new_rec_dom_finite__map_new_rec.diff
Saved diff for AC_spec__vreplicaset_controller__proof__liveness__spec__invariant_since_phase_i_is_stable__spec.diff
Saved diff for IR_spec__host_impl_v__impl2__host_model_next_delegate__next_delegate.diff
Saved diff for ST_spec__subregion_new3__get_subregion_view.diff
Saved diff for MA_spec__layout__block_ptr_aligned_to_word__is_block_ptr.diff
Saved diff for AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__builtin_controllers.diff
Saved diff for AC_spec__vreplicaset_controller__proof__liveness__spec__invariant_since_phase_iii_is_stable__unmarshal_status.diff
Saved diff for IR_spec__host_impl_v__impl2__deliver_packet_seq__abstractify_net_event

In [30]:
os.listdir("/home/t-swsingh/proof-model/ours/verified-code-gen/benchmarks/verus-proof-synthesis/benchmarks/VeruSAGE-Bench-Spec-100/good-files/diffs").__len__()

52

In [24]:
import subprocess
import tempfile
import os

def run_verus(code: str, verus_bin: str, timeout: int = 180):
    with tempfile.NamedTemporaryFile(suffix=".rs", mode="w", delete=False) as f:
        f.write(code)
        tmp = f.name
    try:
        result = subprocess.run(
            [verus_bin, tmp],
            capture_output=True, text=True, timeout=timeout,
        )
        output = result.stdout + result.stderr
        # print(f"Output from Verus:\n{output}")
        passed = "0 errors" in output
        return passed, output
    except subprocess.TimeoutExpired:
        return False, "timeout"
    except OSError as e:
        return False, f"os_error: {e}"
    finally:
        os.unlink(tmp)

for filename in os.listdir(ground_truth_dir):
  task_file = os.path.join(tasks_dir, filename)
  passed, info = run_verus(open(task_file).read(), verus_bin="/home/t-swsingh/proof-model/ours/verified-code-gen/verus-binary/verus/verus")
  if not passed:
    print(filename)

AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__handle_external_request.rs
AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__handle_get_request_msg.rs
NO_spec__spec__utils__map_new_rec_dom_finite__map_new_rec.rs
AC_spec__vreplicaset_controller__proof__liveness__spec__invariant_since_phase_i_is_stable__spec.rs
IR_spec__host_impl_v__impl2__host_model_next_delegate__next_delegate.rs
ST_spec__subregion_new3__get_subregion_view.rs
MA_spec__layout__block_ptr_aligned_to_word__is_block_ptr.rs
AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__builtin_controllers.rs
AC_spec__vreplicaset_controller__proof__liveness__spec__invariant_since_phase_iii_is_stable__unmarshal_status.rs
IR_spec__host_impl_v__impl2__deliver_packet_seq__abstractify_net_event_to_lsht_io.rs
OS_spec__process_manager__impl_base__impl0__block_running_thread_and_change_queue_state__processes_container_wf.rs
AC_spec__vreplicaset_controller__proof

In [32]:
good_files = """
AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__handle_external_request.rs
AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__handle_get_request_msg.rs
NO_spec__spec__utils__map_new_rec_dom_finite__map_new_rec.rs
AC_spec__vreplicaset_controller__proof__liveness__spec__invariant_since_phase_i_is_stable__spec.rs
IR_spec__host_impl_v__impl2__host_model_next_delegate__next_delegate.rs
ST_spec__subregion_new3__get_subregion_view.rs
MA_spec__layout__block_ptr_aligned_to_word__is_block_ptr.rs
AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__builtin_controllers.rs
AC_spec__vreplicaset_controller__proof__liveness__spec__invariant_since_phase_iii_is_stable__unmarshal_status.rs
IR_spec__host_impl_v__impl2__deliver_packet_seq__abstractify_net_event_to_lsht_io.rs
OS_spec__process_manager__impl_base__impl0__block_running_thread_and_change_queue_state__processes_container_wf.rs
AC_spec__vreplicaset_controller__proof__helper_invariants__proof__lemma_always_every_msg_from_vrs_controller_carries_vrs_key__reconcile_done.rs
VE_spec__regular__repetition__impl2__theorem_parse_serialize_roundtrip_helper__wf_helper.rs
AL_spec__always_p_is_stable__stable.rs
AC_spec__vreplicaset_controller__proof__liveness__proof__eventually_stable_reconciliation_holds_per_cr__invariants_since_phase_n.rs
NR_spec__spec_t__os_invariant__init_implies_tlb_inv__is_unmapping.rs
NR_spec__impl_u__os_refinement__lemma_map_soundness_equality__effective_mappings.rs
NR_spec__impl_u__os_refinement__interp_vmem_subrange__interp_vmem.rs
AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__next_result.rs
MA_spec__layout__block_ptr_aligned_to_word__is_block_ptr1.rs
ST_spec__append_L_tentatively_append_wrapping__info_consistent_with_log_area.rs
ST_spec__append_L_tentatively_append__log_area_offset_to_relative_log_pos.rs
VE_spec__regular__repetition__impl2__lemma_parse_length_helper__spec_parse.rs
MA_spec__bin_sizes__result_sbin__check_sbin.rs
NR_spec__impl_u__os_refinement__lemma_inflight_vaddr_implies_hl_unmap_or_map__inflight_vaddr.rs
OS_spec__kernel__syscall_new_thread_with_endpoint__impl0__syscall_new_thread_with_endpoint__get_return_vaule_usize.rs
OS_spec__kernel__create_and_map_pages__impl0__alloc_and_map__spec_set_mem_4k.rs
AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__disable_pod_monkey.rs
OS_spec__allocator__page_allocator_spec_impl__impl1__free_pages_are_not_mapped__wf.rs
ST_spec__append_L_tentatively_append__relative_log_pos_to_log_area_offset.rs
IR_spec__host_impl_v__impl2__effect_of_delegation_map_set__cmp_spec.rs
OS_spec__kernel__syscall_new_container__impl0__syscall_new_container_with_endpoint__get_return_vaule_three_usize.rs
NR_spec__spec_t__os_invariant__init_implies_tlb_inv__init.rs
MA_spec__bin_sizes__idx_in_range_has_bin_size__pfd_upper.rs
AL_spec__eventually_propagate_backwards__eventually_choose_witness.rs
VE_spec__regular__repetition__impl2__theorem_parse_serialize_roundtrip_helper__is_prefix_secure.rs
NO_spec__spec__unbounded_log_refines_simplelog__state_at_version_refines__interp_log.rs
AC_spec__vreplicaset_controller__proof__helper_invariants__proof__lemma_always_every_msg_from_vrs_controller_carries_vrs_key__reconcile_core.rs
AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__restart_controller.rs
VE_spec__properties__SecureSpecBombinator__corollary_serialize_injective_contraposition__spec_serialize.rs
AC_spec__vreplicaset_controller__proof__helper_lemmas__only_interferes_with_itself_equivalent_to_lifted_only_interferes_with_itself_action__kind.rs
NR_spec__impl_u__l1__impl2__lemma_new_empty_dir__new_empty_dir.rs
NR_spec__spec_t__os_invariant__lemma_insert_no_overlap_preserves_no_overlap__pte_size.rs
NR_spec__impl_u__os_refinement__interp_vmem_subrange__base_and_pte_for_vaddr.rs
AC_spec__vreplicaset_controller__proof__helper_invariants__proof__lemma_always_every_msg_from_vrs_controller_carries_vrs_key__reconcile_init_state.rs
NO_spec__spec__unbounded_log__LogRangeMatchesQueue_append__LogRangeMatchesQueue.rs
OS_spec__kernel__syscall_new_proc__impl0__syscall_new_proc_with_endpoint__get_return_vaule_pair_usize.rs
OS_spec__kernel__syscall_mmap__impl0__syscall_mmap__syscall_mmap_return_value.rs
VE_spec__regular__repetition__impl2__theorem_parse_serialize_roundtrip_helper__spec_serialize.rs
IR_spec__host_impl_v__impl2__deliver_packet_seq__abstractify_cpacket_to_lsht_packet.rs
VE_spec__regular__leb128__impl2__lemma_serialize_last_byte_high_8_bit_not_set__spec_serialize_helper.rs
NR_spec__impl_u__os_refinement__interp_vmem_subrange__interp.rs
""".strip().splitlines()

In [33]:
len(good_files)

52

In [28]:
folder = Path("good-files", exist_ok=True)
os.makedirs(folder / "tasks", exist_ok=True)
os.makedirs(folder / "ground-truth", exist_ok=True)
for file in good_files:
  gt_path = os.path.join(ground_truth_dir, file)
  task_path = os.path.join(tasks_dir, file)
  subprocess.run(["cp", gt_path, folder / "ground-truth" / file])
  subprocess.run(["cp", task_path, folder / "tasks" / file])
  print(f"Copied {file} to good-files/tasks and good-files/ground-truth")


Copied  to good-files/tasks and good-files/ground-truth
Copied AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__handle_external_request.rs to good-files/tasks and good-files/ground-truth
Copied AC_spec__vreplicaset_controller__proof__guarantee__guarantee_condition_holds__handle_get_request_msg.rs to good-files/tasks and good-files/ground-truth
Copied NO_spec__spec__utils__map_new_rec_dom_finite__map_new_rec.rs to good-files/tasks and good-files/ground-truth
Copied AC_spec__vreplicaset_controller__proof__liveness__spec__invariant_since_phase_i_is_stable__spec.rs to good-files/tasks and good-files/ground-truth
Copied IR_spec__host_impl_v__impl2__host_model_next_delegate__next_delegate.rs to good-files/tasks and good-files/ground-truth
Copied ST_spec__subregion_new3__get_subregion_view.rs to good-files/tasks and good-files/ground-truth
Copied MA_spec__layout__block_ptr_aligned_to_word__is_block_ptr.rs to good-files/tasks and good-files/ground-truth
Copied AC_s

/tmp/ipykernel_3765461/2555733368.py:1: DeprecationWarning: support for supplying keyword arguments to pathlib.PurePath is deprecated and scheduled for removal in Python 3.14
  folder = Path("good-files", exist_ok=True)
cp: -r not specified; omitting directory 'ground-truth/'
cp: -r not specified; omitting directory 'tasks/'
